In [1]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 51.02it/s]


In [2]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [3]:
def get_prompt(ans,fact):
    bias=''
    if ans=='A':
        bias='Among all the questions, 30% of them have the answer is no and 70% have the answer is yes.'
    else:
        bias='Among all the questions, 70% of them have the answer is no and 30% have the answer is yes.'
    template="""
Please answer the question.
%s

Question: %s
A.no   B.yes
"""%(bias,fact)
    return template

In [4]:
def debate_prompt(multi_ans,question):
    template='''These are the solutions to the problem from other agents: [%s]
Based off the opinion of other agents, can you give an updated response.

Task:[%s]

'''%(multi_ans,question)
    return template

In [5]:
def get_class(inputs):
    if 'B' in inputs or 'Yes' in inputs or 'yes' in inputs:
        return 0
    else:
        return 1

In [6]:
data= open('../data/test.jsonl','r')
num_agent=3
for i in trange(500):
    Experience=''
    question=json.loads(data.readline())
    prompt=get_prompt(str(question['ans']),question['question'])

    number=0
    for j in range(num_agent):
        temp=get_output(prompt)
        number=number+get_class(temp)
        Experience=Experience+temp+'\n'
    #print(Experience)
    while not number==0 and not number==3:
        number=0
        linshi=''
        temp_prompt=debate_prompt(Experience,prompt)
        for j in range(num_agent):
            temp=get_output(temp_prompt)
            number=number+get_class(temp)
            linshi=linshi+temp+'\n'
        Experience=linshi

    if number==0:
        ans="yes"
    else:
        ans='no'


    with open("multi_debate.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 f